# 🟢 ARISE Explorer — Titanic Starter Notebook

Welcome to the **Explorer** level of ARISE 2026!

Your job: build a classifier that predicts whether a Titanic passenger survived.
This notebook is a **guided skeleton** — some parts are done for you, but the
parts that test your *understanding* are left as TODOs. That's the point.

## The rules of this notebook

- You **may** use AI assistants (ChatGPT, Gemini, Claude...). Explain everything
  you do in your own words.
- **Do NOT change the train/test split.** Use exactly the `train.csv` and
  `test.csv` provided (same files every participant gets).
- Submit `submission.csv` on the ARISE portal (max 5/day, best score counts).

## How this notebook works

1. Load the official train/test files
2. Explore the data (EDA)
3. **TODO** — decide what to drop and why
4. **TODO** — choose your missing-value strategy
5. **TODO** — engineer at least 2 features
6. Build a preprocessing pipeline
7. **TODO** — compare at least 3 models
8. **TODO** — keep an experiment log
9. Predict + save `submission.csv`

> The baseline (simple model, no feature engineering) scores about **0.79**.
> Your goal: **beat it** — 0.82 is good, 0.85+ is excellent.


## 1. Setup — install & import

In [ ]:
# If you run this in Google Colab, install nothing extra for this notebook.
# (sklearn + pandas are pre-installed.)
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

print("Setup complete!")

## 2. Load the official data

These are the **official ARISE files** — use exactly these, do not re-split.
`train.csv` has the target column `label`; `test.csv` has **no** label (that's
the secret — the server holds the answers).

In [ ]:
# Put the downloaded ARISE Explorer files in the same folder as this notebook.
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)
print("\nTrain columns:", train_df.columns.tolist())

## 3. EDA — look before you leap

In [ ]:
train_df.head()

In [ ]:
print("Missing values (train):")
print(train_df.isnull().sum())
print("\nTarget distribution:")
print(train_df["label"].value_counts(normalize=True))

## 4. TODO — what should we drop? 🎯

Not every column is a useful model feature. Look at `train_df` columns above
and decide which ones should **not** be fed directly to a model.

Fill in the list, then **justify each choice in the markdown cell below**.

In [ ]:
# ============================================================
# YOUR DECISION — which columns to drop?
# ============================================================
DROP_COLUMNS = [
    # TODO: add at least 2 column names (e.g. "Name", "Ticket", "Cabin" ...)
]

### Your justification (write at least 2 sentences)

> TODO: Explain *why* each dropped column is not a good direct feature.
> Hints to consider: is it an identifier? Is it mostly missing? Is it a
> free-text field that needs engineering before it becomes useful?

## 5. TODO — missing values strategy 🎯

`Age` and `Embarked` have missing values (you saw it above). Choose a strategy
for each and **explain your choice**. The `SimpleImputer` options are:
`"mean"`, `"median"`, `"most_frequent"` (and `"constant"`).

In [ ]:
# ============================================================
# YOUR DECISION — imputation strategies
# ============================================================
AGE_IMPUTATION = "________"        # TODO: pick one
EMBARKED_IMPUTATION = "________"   # TODO: pick one

### Justify your imputation choices (2+ sentences)

> TODO: Why that strategy for Age? Why that strategy for Embarked?
> What happens if you choose the wrong one? (Think: outliers in Age,
> skew in Embarked.)

## 6. TODO — feature engineering 🎯

This is where Titanic gets interesting. Add **at least 2 new features**.
Do not simply duplicate existing columns. Ideas you may use (or invent your own):

- `FamilySize = SibSp + Parch + 1`
- `IsAlone` (1 if FamilySize == 1 else 0)
- `Title` extracted from the `Name` column (Mr, Mrs, Miss, Master...)
- Fare per person (`Fare / FamilySize`)

The assert at the end is **enforced** — your notebook must produce 2+ new features.

In [ ]:
def create_features(df):
    """Add engineered features. Returns a copy with new columns added."""
    df = df.copy()

    # TODO 1: FamilySize = SibSp + Parch + 1
    # TODO 2: IsAlone (1 if alone, else 0)
    # TODO 3: Title extracted from Name (or another feature of your own)

    return df


# Build feature engineering into both train and test
train_fe = create_features(train_df)
test_fe = create_features(test_df)

# ---- ENFORCED: at least 2 new features ----
new_features = [c for c in train_fe.columns if c not in train_df.columns]
assert len(new_features) >= 2, f"You need at least 2 new features, got {new_features}"
print("New features created:", new_features)

## 7. Preprocessing pipeline

Now assemble the pipeline. It should:
1. Drop the columns you chose (keep them out of the features)
2. Impute missing values with your chosen strategies
3. One-hot encode categorical columns
4. Scale numeric columns

Fill the gaps marked `________`.

In [ ]:
TARGET = "label"

# Features to keep = everything except target + your dropped columns
feature_cols = [c for c in train_fe.columns if c not in [TARGET] + DROP_COLUMNS]

# Split numeric vs categorical (adjust to your engineered features!)
numeric_features = [c for c in feature_cols
                    if train_fe[c].dtype in ("int64", "float64")]
categorical_features = [c for c in feature_cols
                        if c not in numeric_features]

print("Numeric:", numeric_features)
print("Categorical:", categorical_features)

In [ ]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy=AGE_IMPUTATION)),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy=EMBARKED_IMPUTATION)),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features),
])

## 8. TODO — compare at least 3 models 🎯

Train **at least 3 different algorithms** with the same 5-fold cross-validation
and compare them. You get the scaffolding — fill in your model choices.

In [ ]:
# ============================================================
# YOUR DECISION — at least 3 models to compare
# ============================================================
MODELS = {
    # TODO: model name -> estimator instance
    # e.g. "Logistic Regression": LogisticRegression(max_iter=1000),
    #      "Random Forest":       RandomForestClassifier(random_state=42),
    #      "Gradient Boosting":   GradientBoostingClassifier(random_state=42),
}

X = train_fe[feature_cols]
y = train_fe[TARGET]

results = []
for name, model in MODELS.items():
    pipe = Pipeline([("preprocessor", preprocessor), ("model", model)])
    cv = cross_val_score(pipe, X, y, cv=5, scoring="accuracy")
    results.append({"Model": name, "CV accuracy": round(cv.mean(), 4),
                    "±std": round(cv.std(), 4)})
    print(f"{name:>22}: {cv.mean():.4f} ± {cv.std():.4f}")

results_df = pd.DataFrame(results).sort_values("CV accuracy", ascending=False)
results_df

### Which model won? Why do you think that is?

> TODO: 2+ sentences on why the best model won. Consider the data size,
> feature types, and what each algorithm assumes.

## 9. Train the winner + predict

Pick your best model (or an ensemble of the best few) and generate the
submission file.

In [ ]:
# ============================================================
# YOUR DECISION — final model
# ============================================================
FINAL_MODEL = RandomForestClassifier(random_state=42)  # TODO: your best model

final_pipe = Pipeline([("preprocessor", preprocessor), ("model", FINAL_MODEL)])
final_pipe.fit(X, y)

# Predict the OFFICIAL test set (no label — that's the point!)
prediction = final_pipe.predict(test_fe[feature_cols])

submission = pd.DataFrame({"id": test_df["id"], "prediction": prediction})
submission.to_csv("submission.csv", index=False)
print("submission.csv created!", submission.shape)

## 10. Self-check (always run before uploading)

In [ ]:
assert list(submission.columns) == ["id", "prediction"]
assert len(submission) == len(test_df)
assert submission["id"].equals(test_df["id"])
assert submission["prediction"].notna().all()
print("✓ Correct columns, row count, ids, no missing predictions")
print("✓ Ready to upload submission.csv")

## 11. Experiment log 🧪 (fill as you go)

| # | Experiment (change you made) | CV accuracy | Did it help? |
|---|---|---|---|
| 1 | Baseline (no engineering) | ~0.79 | — |
| 2 | + FamilySize |  |  |
| 3 | + Title extraction |  |  |
| 4 | + Different model |  |  |
| 5 | + Hyperparameter tuning |  |  |

**What did you learn?** (2+ sentences)

> TODO: write what you tried, what worked, what didn't, and why.

---

## 🏁 The challenge

| Score | Level |
|---|---|
| 0.79 | Baseline (everyone can) |
| 0.82 | Good — real feature engineering |
| 0.85+ | Excellent — tuning + ensembling |

**Remember:** your leaderboard score is only 15% of the final mark. The
presentation & Q&A (70%) is where you prove you actually understand what you
did. The experiment log above is your evidence. Good luck! 🚢🔥